# DIP super-resolution — experiment runner (Set5 ×4, tracked in W&B)

Runs DIP super-resolution experiments from `lakshjain7/deep-image-prior` (`super_resolution_cli.py`). The paper's regularizers (TV, nuclear norm, TVSSAH, backward diffusion) are ported unchanged from the authors' code (`capo-urjc/dip-sisr@e2c80f0`) into `utils/regularizers.py`. Weights are the paper's Table 3 values for Set5.

**How to use**
1. Runtime → Change runtime type → **GPU**. Add a Colab secret `WANDB_API_KEY` with notebook access.
2. Run sections 1, 2, 4. Set `EXPERIMENT` in section 3 and run it. It prints how many runs are new and a time estimate.
3. Sections 5–6 are one-time checks (already passed); skip them on later sessions.
4. Run section 7. Every configuration is one W&B run; finished ones are skipped, so after a disconnect re-run 1–4 and 7.
5. For tables and interpretation, use **`DIP_Results_Summary.ipynb`** (no GPU needed).

**Never run the same `EXPERIMENT` in two notebooks at once** — split them (e.g. `mismatch-A` in one, `mismatch-B` in the other).

Training setups: **adapted** = lr 0.01, input noise 0.03, 2000 iterations (our earlier settings). **paper** = lr 0.001, noise 0.01, 5500 iterations.

## 1. Setup

In [ ]:
import os, sys, subprocess, importlib, json, time
from pathlib import Path

REPO_URL = "https://github.com/lakshjain7/deep-image-prior.git"
BRANCH = "claude/epic-ride-rnqbk4"
REPO_DIR = Path("/content/deep-image-prior")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "merge", "--ff-only", f"origin/{BRANCH}"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))

for mod, pkg in [("wandb", "wandb"), ("skimage", "scikit-image"), ("cv2", "opencv-python-headless"), ("pytest", "pytest")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch, wandb, pandas as pd
from IPython.display import display
print("repo commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"

## 2. W&B login (key comes from Colab Secrets, never typed here)

In [ ]:
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception as e:
    print("No Colab secret found -> interactive login")
wandb.login()
print("W&B entity:", wandb.Api().default_entity)

## 3. Choose the experiment

| `EXPERIMENT` | What it runs | Status |
|---|---|---|
| `matrix` | 5 images × 3 losses × 5 regularizer conditions, PIL-Lanczos LR (75 runs) | done |
| `operator` | L1, no regularizer; LR made by PIL Lanczos or bicubic × model kernel Lanczos-2/3 (20 runs) | done (1 cell missing) |
| `paper-test` | paper training settings, bicubic LR, butterfly + head, none/TV (4 runs) | done |
| `mismatch-A` | L1, none vs TV, 5 images, LR from PIL Lanczos / antialiased bicubic / bicubic (30 runs, ~16 new) | **to run** |
| `mismatch-B` | bicubic LR, L1, NN / TVSSAH / BD, 4 images (12 runs) | **to run** |
| `mismatch` | A + B together | **to run** |

In [ ]:
import stage3
from stage3 import IMAGES, LOSSES, REGULARIZERS

EXPERIMENT = "mismatch"          # see the table above
PROJECT = "dip-sr-regularizers"
SEEDS = [7]
TIME_BUDGET_MIN = 150            # stop starting new runs after this many minutes in one session
W = dict(stage3.DEFAULT_REG_WEIGHTS)

def make_jobs(name):
    if name == "matrix":
        return "adapted", stage3.build_jobs(IMAGES, LOSSES, REGULARIZERS, SEEDS, "adapted", W)
    if name == "operator":
        return "adapted", [j for m in ["pil-lanczos", "bicubic"] for k in ["lanczos2", "lanczos3"]
                           for j in stage3.build_jobs(IMAGES, ["l1"], ["none"], SEEDS, "adapted", W, m, k)]
    if name == "paper-test":
        return "paper", stage3.build_jobs(["butterfly", "head"], ["l1"], ["none", "tv"], SEEDS, "paper", W, "bicubic")
    a = [j for m in ["pil-lanczos", "bicubic-aa", "bicubic"]
         for j in stage3.build_jobs(IMAGES, ["l1"], ["none", "tv"], SEEDS, "adapted", W, m)]
    b = stage3.build_jobs(["bird", "butterfly", "head", "woman"], ["l1"], ["nn", "tvssah", "bd"], SEEDS, "adapted", W, "bicubic")
    return "adapted", {"mismatch-A": a, "mismatch-B": b, "mismatch": a + b}[name]

PROTOCOL, jobs = make_jobs(EXPERIMENT)
P = stage3.PROTOCOLS[PROTOCOL]
CFG = {"factor": 4, "lr": P["lr"], "reg_noise_std": P["reg_noise_std"], "num_iter": P["num_iter"],
       "log_freq": 100, "protocol_note": P["note"], "tags": [PROTOCOL, EXPERIMENT]}
OUT = Path("/content/dip_outputs"); (OUT / "logs").mkdir(parents=True, exist_ok=True)
tracker = stage3.Tracker(None, PROJECT, OUT / "runs")

todo = [j for j in jobs if not tracker.is_complete(j["key"])]
scale = CFG["num_iter"] / 2000
est = sum((10 if j["image"] == "baby" else 3) * scale for j in todo)
print(f"{EXPERIMENT}: {len(jobs)} configurations, {len(jobs) - len(todo)} already done, {len(todo)} to run")
print(f"Estimated GPU time: ~{est:.0f} min ({PROTOCOL}: lr {CFG['lr']}, noise {CFG['reg_noise_std']}, {CFG['num_iter']} iterations)")

## 4. Set5 HR images

In [ ]:
import urllib.request, tarfile
SET5_DIR = Path("/content/Set5")
try:
    image_paths = stage3.find_set5(SET5_DIR)
except FileNotFoundError:
    SET5_DIR.mkdir(parents=True, exist_ok=True)
    url = "https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz"
    tmp = SET5_DIR / "Set5_HR.tar.gz"
    urllib.request.urlretrieve(url, tmp)
    tarfile.open(tmp).extractall(SET5_DIR)
    image_paths = stage3.find_set5(SET5_DIR)
for k, v in image_paths.items(): print(f"{k:10s} {v}")

If this cell fails, upload the five HR Set5 images to `/content/Set5/` and re-run it.

## 5. (One-time) Check the regularizers match the authors' code

In [ ]:
ref = Path("/content/dip-sisr-ref")
if not ref.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/capo-urjc/dip-sisr.git", str(ref)], check=True)
    subprocess.run(["git", "-C", str(ref), "checkout", "-q", "e2c80f0e60880a868e77dd024b70cb4a2e4b3ed5"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_regularizers.py"],
                   env=dict(os.environ, DIP_SISR_DIR=str(ref)), capture_output=True, text=True)
print(r.stdout[-500:]); assert r.returncode == 0, "Regularizers do not match the official code"

## 6. (One-time) Smoke test: one short regularized run (200 iterations), logged as a validation run

In [ ]:
import copy
sj = copy.deepcopy(next(j for j in jobs if j["reg"] != "none"))
sj["key"] = "smoke__" + sj["key"]
SCFG = dict(CFG, num_iter=200, job_type="smoke", tags=["smoke", "validation"], name_prefix="smoke")
if not tracker.is_complete(sj["key"]):
    rec = stage3.run_job(sj, SCFG, image_paths[sj["image"]], tracker, OUT / "logs")
    assert rec["status"] == "complete", rec.get("error_tail")
print("smoke run complete; resume check ->", stage3.run_job(sj, SCFG, image_paths[sj["image"]], tracker, OUT / "logs")["status"])

## 7. Run (resumable)

Each configuration is one W&B run (name `s3_<image>_<loss>_<reg>_seed<seed>`, tags for loss, regularizer, image, LR input). A crash marks the run failed in W&B and the configuration is retried next time. Interrupting the cell is safe.

In [ ]:
t0, done_now, durations, failures = time.time(), 0, [], []
for n, job in enumerate(jobs, 1):
    spent = (time.time() - t0) / 60
    avg = sum(durations) / len(durations) / 60 if durations else 0
    if spent + 1.2 * avg > TIME_BUDGET_MIN:
        print(f"Time budget reached ({spent:.0f} min). Re-run this cell in a new session to continue."); break
    rec = stage3.run_job(job, CFG, image_paths[job["image"]], tracker, OUT / "logs")
    if rec["status"] == "skipped_complete":
        print(f"[{n}/{len(jobs)}] skip      {job['key']}"); continue
    durations.append(rec["elapsed_s"]); done_now += rec["status"] == "complete"
    print(f"[{n}/{len(jobs)}] {rec['status']:9s} {job['key']}  ({rec['elapsed_s']/60:.1f} min)")
    if rec["status"] != "complete":
        failures.append(rec); print("   ", rec.get("error_tail", "")[-300:].replace("\n", "\n    "))
        with open(OUT / "failures.jsonl", "a") as f: f.write(json.dumps(rec) + "\n")
print(f"This session: {done_now} completed, {len(failures)} failed.")

## 8. Progress check (full tables: `DIP_Results_Summary.ipynb`)

In [ ]:
df = stage3.collect_results(jobs, tracker)
print(f"{EXPERIMENT}: complete {int((df.status == 'complete').sum())}/{len(df)}")
display(df[["image", "loss", "reg", "lr_method", "kernel", "status", "psnr_hr", "ssim_hr"]].round(3))